In [1]:
import os
import glob
import pickle
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score, recall_score,precision_score
from tensorflow.keras.models import load_model
import tensorflow as tf
from tensorflow.keras.saving import register_keras_serializable
import matplotlib.pyplot

2025-07-31 14:02:08.598792: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-31 14:02:08.619825: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753970528.643033    1861 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753970528.649995    1861 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1753970528.668344    1861 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [2]:
cache_dir = "/home/onyxia/work/DATA/PRED2"

xtrain_path = os.path.join(cache_dir, 'X_train.npy')
ytrain_path = os.path.join(cache_dir, 'y_train.npy')
xtest_path  = os.path.join(cache_dir, 'X_test.npy')
ytest_path  = os.path.join(cache_dir, 'y_test.npy')
# x_vrai_test_path  = os.path.join(cache_dir, 'X_vrai_test.npy')
# y_vrai_test_path  = os.path.join(cache_dir, 'y_vrai_test.npy')

X_train = np.load(xtrain_path)
y_train = np.load(ytrain_path)
X_test  = np.load(xtest_path)
y_test  = np.load(ytest_path)
# X_vrai_test  = np.load(x_vrai_test_path)
# y_vrai_test  = np.load(y_vrai_test_path)

In [3]:

import pandas as pd

df_X_train = pd.DataFrame(X_train)
print(df_X_train.head())

    0    1    2    3    4    5    6    7    8    9   ...   14   15   16   17  \
0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...  0.0  0.0  0.0  0.0   
1  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...  0.0  0.0  0.0  0.0   
2  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...  0.0  0.0  0.0  0.0   
3  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...  0.0  0.0  0.0  0.0   
4  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...  0.0  0.0  0.0  0.0   

    18   19   20   21   22   23  
0  0.0  0.0  0.0  0.0  0.0  0.0  
1  0.0  0.0  0.0  0.0  0.0  0.0  
2  0.0  0.0  0.0  0.0  0.0  0.0  
3  0.0  0.0  0.0  0.0  0.0  0.0  
4  0.0  0.0  0.0  0.0  0.0  0.0  

[5 rows x 24 columns]


In [4]:
import numpy as np
from sklearn.metrics import confusion_matrix

def print_class_metrics(y_true, y_pred, labels=[0,1,2]):
    """
    Affiche, pour chaque classe dans `labels` :
      - support
      - precision
      - rappel
      - f1-score
    Et l'accuracy globale.
    """
    # matrice de confusion (lignes = y_true, colonnes = y_pred)
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    
    # overall accuracy
    acc = np.mean(np.array(y_true) == np.array(y_pred))
    print(f"Accuracy globale : {acc:.4f}\n")
    
    # metrics par classe
    for i, lbl in enumerate(labels):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        support = cm[i, :].sum()
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1        = (2 * precision * recall / (precision + recall)
                     if (precision + recall) > 0 else 0.0)
        
        print(f"Classe {lbl}:")
        print(f"  support   = {support}")
        print(f"  precision = {precision:.3f}")
        print(f"  rappel    = {recall:.3f}")
        print(f"  f1-score  = {f1:.3f}\n")

In [5]:
@register_keras_serializable()
class GRUWithTimeMajor(tf.keras.layers.GRU):
    def __init__(self, *args, time_major=False, **kwargs):
        super().__init__(*args, **kwargs)


model_path_GRU = "/home/onyxia/work/DATA/perf"



model_files_GRU_CNN = glob.glob(os.path.join(model_path_GRU, 'modele_que_volant_CNN0.h5'))
model_file_GRU_CNN = model_files_GRU_CNN[0]

model_GRU_CNN = tf.keras.models.load_model(
            model_file_GRU_CNN,
            custom_objects={'GRU': GRUWithTimeMajor}
        )



I0000 00:00:1753970534.382820    1861 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13282 MB memory:  -> device: 0, name: NVIDIA A2, pci bus id: 0000:ca:00.0, compute capability: 8.6


In [6]:
y_pred_proba_GRU_CNN = model_GRU_CNN.predict(X_test)
y_pred_GRU_CNN = np.argmax(y_pred_proba_GRU_CNN, axis=1)
print_class_metrics(y_test, y_pred_GRU_CNN)
c = confusion_matrix(y_test, y_pred_GRU_CNN, labels=[0,1,2])
print(f"FN classe no : {c[0,1]+c[0,2]}")



# y_pred_proba_GRU_CNN2 = model_GRU_CNN.predict(X_vrai_test)
# y_pred_GRU_CNN2 = np.argmax(y_pred_proba_GRU_CNN2, axis=1)

# print_class_metrics(y_vrai_test, y_pred_GRU_CNN2)
# c = confusion_matrix(y_vrai_test, y_pred_GRU_CNN2, labels=[0,1,2])
# print(f"FN classe no : {c[0,1]+c[0,2]}")

I0000 00:00:1753970536.044466  550693 cuda_dnn.cc:529] Loaded cuDNN version 90501


7086/7086 ━━━━━━━━━━━━━━━━━━━━ 21s 3ms/step
Accuracy globale : 0.8683

Classe 0:
  support   = 191677
  precision = 0.966
  rappel    = 0.894
  f1-score  = 0.928

Classe 1:
  support   = 17566
  precision = 0.598
  rappel    = 0.749
  f1-score  = 0.665

Classe 2:
  support   = 17478
  precision = 0.453
  rappel    = 0.709
  f1-score  = 0.553

FN classe no : 20359


In [7]:
# import numpy as np
# import xgboost as xgb
# from xgboost import XGBClassifier
# from sklearn.model_selection import train_test_split, RandomizedSearchCV
# from sklearn.metrics import classification_report


# aligned_extend = extend_aligned(aligned,2)
# biais = 0.5
# prob_peak = np.zeros_like(y_pred_proba_GRU_CNN)
# prob_peak[:, 1] = aligned_extend * biais
# prob_peak[:, 2] = aligned_extend * biais

# prob_peak[:, 0] = 1 -prob_peak[:, 1] -prob_peak[:, 2]


# # --- 1) Préparation des features meta et labels ---
# # On part de y_pred_proba (N×3) et aligned (N,)
# X_meta = np.column_stack([y_pred_proba_GRU_CNN, prob_peak])
# y_meta = y_test

# # --- 2) Séparation en train / validation / test ---
# # 20% pour le test final
# X_temp, X_testt, y_temp, y_testt = train_test_split(
#     X_meta, y_meta,
#     test_size=0.20,
#     random_state=42,
#     shuffle=False
    
# )
# # Parmi les 80% restants, 25% pour la validation (soit 20% du total)
# X_train, X_val, y_train, y_val = train_test_split(
#     X_temp, y_temp,
#     test_size=0.25,
#     random_state=42,
#     shuffle=False
# )

# print("Shapes:")
# print("  Train :", X_train.shape, y_train.shape)
# print("  Val   :", X_val.shape,   y_val.shape)
# print("  Test  :", X_testt.shape,  y_testt.shape)

# # --- 3) Recherche aléatoire d'hyperparamètres sur train/val ---
# param_dist = {
#     'n_estimators':    [100, 200, 300],
#     'learning_rate':   [0.01, 0.1, 0.2],
#     'max_depth':       [3, 5, 7],
#     'subsample':       [0.7, 0.8, 1.0],
#     'colsample_bytree':[0.7, 0.8, 1.0],
# }
# xgb_base = XGBClassifier(
#     use_label_encoder=False,
#     eval_metric='mlogloss',
#     random_state=42,
#     n_jobs=-1
# )
# # On construit un seul split (train indices, val indices) pour RandomizedSearchCV
# idx_train = np.arange(len(X_train))
# idx_val   = np.arange(len(X_val)) + len(X_train)
# search = RandomizedSearchCV(
#     xgb_base,
#     param_dist,
#     n_iter=20,
#     scoring='f1_macro',
#     cv=[(idx_train, idx_val)],
#     random_state=42,
#     n_jobs=-1,
#     verbose=1
# )
# # Concatène train+val pour que cv fonctionne
# X_tr_val = np.vstack([X_train, X_val])
# y_tr_val = np.hstack([y_train, y_val])
# search.fit(X_tr_val, y_tr_val)
# best_xgb = search.best_estimator_
# print("Meilleurs params (validation) :", search.best_params_)
# print("Score F1-macro (validation)  :", search.best_score_)

# # --- 4) Entraînement final avec early stopping via xgb.train ---
# # Création des DMatrix
# dtrain = xgb.DMatrix(X_tr_val, label=y_tr_val)
# dval   = xgb.DMatrix(X_val,    label=y_val)
# dtest  = xgb.DMatrix(X_testt,   label=y_testt)

# # Paramètres à passer à l'API bas-niveau
# params = best_xgb.get_xgb_params()
# params.update({
#     'objective':  'multi:softprob',
#     'num_class':  3,
#     'eval_metric':'mlogloss'
# })
# # Entraînement avec early stopping
# bst = xgb.train(
#     params,
#     dtrain,
#     num_boost_round=best_xgb.get_params()['n_estimators'],
#     evals=[(dval, 'validation')],
#     early_stopping_rounds=10,
#     verbose_eval=False
# )

# # --- 5) Évaluation sur le test final ---
# y_prob_test = bst.predict(dtest)
# y_pred_test = np.argmax(y_prob_test, axis=1)
# print("\nRésultats sur le jeu de test :")
# print(classification_report(y_testt, y_pred_test, digits=4))
# print(confusion_matrix(y_testt,y_pred_test,labels=[0,1,2]))


In [8]:
# df_comparaison = pd.DataFrame({
#     'y_test': y_testt,
#     'y_pred': y_pred_test
# })

# from IPython.display import display, HTML


# # Affichage avec scroll
# display(HTML(df_comparaison.to_html(max_rows=200, max_cols=3, notebook=True)))

In [9]:
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.utils import resample
from scipy.stats import bootstrap




def bootstrap_per_class(metric_fn, y_true, y_pred, n_classes, n_resamples=500, alpha=0.05):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    results = {}

    for cls in range(n_classes):
        y_true_bin = (y_true == cls).astype(int)
        y_pred_bin = (y_pred == cls).astype(int)

        def stat_fn(indices):
            return metric_fn(y_true_bin[indices], y_pred_bin[indices])

        res = bootstrap((np.arange(len(y_true_bin)),), stat_fn,
                        confidence_level=1-alpha,
                        n_resamples=n_resamples,
                        method='percentile',
                        random_state=42)

        score = metric_fn(y_true_bin, y_pred_bin)
        results[cls] = {
            'score': score,
            'ci_low': res.confidence_interval.low,
            'ci_high': res.confidence_interval.high
        }

    return results


In [10]:
import numpy as np
import pandas as pd
from scipy.stats import mode



def post_substitution(seq):
    """
    Applique la règle :
    Si on observe 3 blocs successifs de même valeur, 
    remplacer le bloc central par l'autre valeur.
    """
    seq = seq.copy()
    i = 0
    n = len(seq)
    blocks = []

    # Découpe les blocs non nuls
    while i < n:
        if seq[i] != 0:
            start = i
            val = seq[i]
            while i < n and seq[i] == val:
                i += 1
            end = i
            blocks.append((start, end, val))
        else:
            i += 1

    # Appliquer la règle sur les triplets
    for j in range(1, len(blocks) - 1):
        prev_val = blocks[j - 1][2]
        curr_val = blocks[j][2]
        next_val = blocks[j + 1][2]

        if prev_val == curr_val == next_val:
            opposite = 1 if curr_val == 2 else 2
            start, end, _ = blocks[j]
            seq[start:end] = opposite

    return seq



def traiter_sequence(seq):
    seq = np.array(seq)
    result = np.zeros_like(seq)
    i = 0

    while i < len(seq):
        if seq[i] != 0:
            start = i
            while i < len(seq) and seq[i] != 0:
                i += 1
            end = i
            segment = seq[start:end]
            length = end - start

            if 4 <= length <= 8:
                maj = mode(segment, keepdims=True).mode[0]
                # Étendre si nécessaire
                if length < 6:
                    missing = 6 - length
                    if start >= missing and all(seq[start - missing:start] == 0):
                        result[start - missing:start] = maj
                        result[start:end] = maj
                    elif end + missing <= len(seq) and all(seq[end:end + missing] == 0):
                        result[start:end] = maj
                        result[end:end + missing] = maj
                    else:
                        continue  # impossible à étendre
                elif length == 6:
                    result[start:end] = maj
                else:
                    # Réduction
                    segment_mod = segment.copy()
                    excess = length - 6
                    for _ in range(excess):
                        if segment_mod[0] != maj:
                            segment_mod = segment_mod[1:]
                            start += 1
                        elif segment_mod[-1] != maj:
                            segment_mod = segment_mod[:-1]
                            end -= 1
                        else:
                            segment_mod = segment_mod[1:]
                            start += 1
                    result[start:end] = maj
        else:
            i += 1

    return post_substitution(result)




In [11]:
y_modif = traiter_sequence(y_pred_GRU_CNN)
# y_modif_xgb = traiter_sequence(y_pred_test)

In [12]:
print_class_metrics(y_test, y_modif)
c = confusion_matrix(y_test, y_modif, labels=[0,1,2])
print(f"FN classe no : {c[0,1]+c[0,2]}")
print(c)
# print_class_metrics(y_testt, y_pred_test)
# c = confusion_matrix(y_testt, y_pred_test, labels=[0,1,2])
# print(f"FN classe no : {c[0,1]+c[0,2]}")
# print(c)

Accuracy globale : 0.8921

Classe 0:
  support   = 191677
  precision = 0.944
  rappel    = 0.940
  f1-score  = 0.942

Classe 1:
  support   = 17566
  precision = 0.629
  rappel    = 0.659
  f1-score  = 0.644

Classe 2:
  support   = 17478
  precision = 0.603
  rappel    = 0.603
  f1-score  = 0.603

FN classe no : 11531
[[180146   5665   5866]
 [  4921  11584   1061]
 [  5780   1159  10539]]


In [13]:
c = confusion_matrix(y_test, y_modif, labels=[0,1,2])
print(c)
c = confusion_matrix(y_test, y_pred_GRU_CNN, labels=[0,1,2])
print(c)

[[180146   5665   5866]
 [  4921  11584   1061]
 [  5780   1159  10539]]
[[171318   7621  12738]
 [  2183  13159   2224]
 [  3859   1226  12393]]


In [14]:
df_comparaison = pd.DataFrame({
    'y_test': y_test,
    'y_pred': y_pred_GRU_CNN,
    'y_modif':y_modif
})

from IPython.display import display, HTML


# Affichage avec scroll
display(HTML(df_comparaison.to_html(max_rows=200, max_cols=3, notebook=True)))

,y_test,y_pred,y_modif
0,0.0,0,0
1,0.0,0,0
2,0.0,0,0
3,0.0,0,0
4,0.0,0,0
5,0.0,0,0
6,0.0,0,0
7,0.0,0,0
8,0.0,0,0
9,0.0,0,0


In [15]:
f1_per_class = bootstrap_per_class(f1_score, y_test, traiter_sequence(y_pred_GRU_CNN), 3)
prec_per_class = bootstrap_per_class(precision_score, y_test, traiter_sequence(y_pred_GRU_CNN), 3)
recall_per_class = bootstrap_per_class(recall_score, y_test, traiter_sequence(y_pred_GRU_CNN), 3)

print("\n--- Intervalles de Confiance par Classe (95%) ---")
for cls in range(3):
    print(f"Classe {cls}:")
    print(f"  F1-score     : {f1_per_class[cls]['score']:.4f} "
          f"(CI: {f1_per_class[cls]['ci_low']:.4f} - {f1_per_class[cls]['ci_high']:.4f})")
    print(f"  Précision    : {prec_per_class[cls]['score']:.4f} "
          f"(CI: {prec_per_class[cls]['ci_low']:.4f} - {prec_per_class[cls]['ci_high']:.4f})")
    print(f"  Rappel       : {recall_per_class[cls]['score']:.4f} "
          f"(CI: {recall_per_class[cls]['ci_low']:.4f} - {recall_per_class[cls]['ci_high']:.4f})")


--- Intervalles de Confiance par Classe (95%) ---
Classe 0:
  F1-score     : 0.9419 (CI: 0.9410 - 0.9427)
  Précision    : 0.9439 (CI: 0.9429 - 0.9449)
  Rappel       : 0.9398 (CI: 0.9386 - 0.9409)
Classe 1:
  F1-score     : 0.6440 (CI: 0.6384 - 0.6498)
  Précision    : 0.6293 (CI: 0.6222 - 0.6365)
  Rappel       : 0.6595 (CI: 0.6528 - 0.6667)
Classe 2:
  F1-score     : 0.6032 (CI: 0.5963 - 0.6094)
  Précision    : 0.6034 (CI: 0.5960 - 0.6111)
  Rappel       : 0.6030 (CI: 0.5951 - 0.6102)


In [16]:

def extract_blocks(seq):
    """Renvoie une liste de tuples (start, end, label) pour les blocs non nuls."""
    seq = np.array(seq)
    blocks = []
    i = 0
    while i < len(seq):
        if seq[i] != 0:
            label = seq[i]
            start = i
            while i < len(seq) and seq[i] == label:
                i += 1
            end = i
            blocks.append((start, end, label))
        else:
            i += 1
    return blocks

def eval_detection(y_true, y_pred, tol=2):
    true_blocks = extract_blocks(y_true)
    pred_blocks = extract_blocks(y_pred)

    matched_true = set()
    correct_label = 0
    wrong_label = 0
    false_positive = 0

    for p_start, p_end, p_label in pred_blocks:
        matched = False
        for i, (t_start, t_end, t_label) in enumerate(true_blocks):
            if abs(p_start - t_start) <= tol:
                matched = True
                if p_label == t_label:
                    correct_label += 1
                else:
                    wrong_label += 1
                matched_true.add(i)
                break
        if not matched:
            false_positive += 1

    return {
        "series_total_y_true": len(true_blocks),
        "series_total_y_pred": len(pred_blocks),
        "correct_label_match": correct_label,
        "wrong_label_match": wrong_label,
        "false_positives": false_positive,
        "missed_true_blocks": len(true_blocks) - len(matched_true)
    }


In [17]:
result = eval_detection(y_test, y_modif, tol=3)
print(result)

{'series_total_y_true': 5841, 'series_total_y_pred': 5961, 'correct_label_match': 4337, 'wrong_label_match': 438, 'false_positives': 1186, 'missed_true_blocks': 1068}


## SON

In [18]:
import os
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import f1_score
import argparse
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


import sys
sys.path.append('/home/onyxia/work/mon-projet/sons')


from torch.utils.data import DataLoader
from archi_rnn_sons import MelSpecWithFeatDataset, CNN  # plus besoin d'importer CRNNBiGRUWithFeat ici



# Liste des IDs à exclure
excluded_ids = ["5684", "19207",  "21939", "22021",  "22103",
                "34289", "34348", "35168", "36020", "37059",  "37541",
                "38626",  "43695", "43835","46959",
                "50821","51185","63851"]


# Charger le dataset complet (ou juste un fichier en filtrant par keep_vids)
dataset = MelSpecWithFeatDataset(
    spec_dir="/home/onyxia/work/DATA/spec",
    feat_dir="/home/onyxia/work/DATA/feat",
    split=None,
    pca_n=30,
    keep_vids= excluded_ids
)

# # Créer les séquences centrées autour de chaque frame
# context = 3
# dataset = CenteredFrameSequenceDataset(dataset, context=context)

# Mettre dans un DataLoader pour le batch
loader = DataLoader(dataset, batch_size=64)

# Charger le modèle complet
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = torch.load("/home/onyxia/work/DATA/spec/strike_cnn_feat_new.pt", map_location=device, weights_only=False)
model.eval()

# Prédiction
all_probs = []
with torch.no_grad():
    for x_seq, feat_seq, _ in loader:
        x_seq, feat_seq = x_seq.to(device), feat_seq.to(device)
        logits = model(x_seq, feat_seq)
        probs = torch.softmax(logits, dim=1)[:, 1]  # probabilité classe 1
        all_probs.extend(probs.cpu().numpy())



import os
import pandas as pd

# Dossier contenant les fichiers .pkl
folder = "/home/onyxia/work/DATA/pkl_merged_peak"





# Récupérer tous les fichiers .pkl dans le dossier
pkl_files = [f for f in os.listdir(folder) if f.endswith(".pkl")]

# Filtrer les fichiers à inclure
included_files = [f for f in pkl_files if not any(excl_id in f for excl_id in excluded_ids)]

# Charger et concaténer les DataFrames
df_list = []
for file in included_files:
    file_path = os.path.join(folder, file)
    try:
        df = pd.read_pickle(file_path)
        df_list.append(df)
    except Exception as e:
        print(f"Erreur lors du chargement de {file}: {e}")

# Concaténer tous les DataFrames
df_concat = pd.concat(df_list, ignore_index=True)

# Affichage du résultat
print(f"{len(df_list)} fichiers chargés et concaténés.")
print(f"Shape du DataFrame final : {df_concat.shape}")

peak_hit_prob = np.zeros(len(df_concat))

# Obtenir les indices où Peak_binary == 1
peak_indices = df_concat.index[df_concat["Peak_binary"] == 1].tolist()

# Assigner les valeurs de all_probs aux bons indices
peak_hit_prob[peak_indices] = all_probs  # Assumes all_probs is aligned with those indices

# Ajouter la colonne au DataFrame
df_concat["peak_hit_prob"] = peak_hit_prob


# Dossier contenant les .npy
folder = "/home/onyxia/work/DATA/spec"

# Ordre désiré (sans extension)
desired_order = [
    "mel_specs_37159",
    "mel_specs_22088",
    "mel_specs_49766",
    "mel_specs_49887",
    "mel_specs_47563",
    "mel_specs_87177",
    "mel_specs_42354",
    "mel_specs_21588",
]

file_lengths = []

for name in desired_order:
    file_path = os.path.join(folder, name + ".npy")
    if not os.path.isfile(file_path):
        raise FileNotFoundError(f"Fichier non trouvé : {file_path}")
    # Charger le tableau et mesurer sa longueur (premier axe)
    arr = np.load(file_path)
    file_lengths.append(arr.shape[0])

print("File lengths:", file_lengths)


# supposons que file_lengths = [L1, L2, …, L8]
file_lengths = [len(df) for df in df_list]

# découpage de votre vecteur brut de labels (226 809)
raw = np.concatenate([peak_hit_prob])  # votre vecteur de y_test avant fenêtrage

aligned = []
start = 0
for L in file_lengths:
    segment = raw[start : start + L]      # extraire les L frames
    segment_trunc = segment[11:]          # retirer les 11 dernières frames
    aligned.append(segment_trunc)
    start += L

aligned = np.concatenate(aligned)  # longueur = 226 721

N = aligned.shape[0]
aligned_extend = np.zeros_like(aligned)
max_dist = 1

for i in range(N):
    # Fenêtre centrée en i, de largeur 2*max_dist+1
    start = max(0, i - max_dist -1)
    end   = min(N, i + max_dist + 1)
    # Copier la valeur non-nulle la plus forte dans la fenêtre
    aligned_extend[i] = np.max(aligned[start:end])
    

8 fichiers chargés et concaténés.
Shape du DataFrame final : (226809, 8)
File lengths: [2324, 3412, 2148, 1927, 3040, 1505, 3296, 1772]


# FUSED

In [19]:
def boosting_with_aligned(y_pred_proba: np.ndarray,
                          aligned: np.ndarray,
                          threshold: float,
                          boost_factor: float) -> np.ndarray:
    """
    Booste les probabilités des classes 1 et 2 là où aligned > threshold.

    Parameters
    ----------
    y_pred_proba : np.ndarray, shape (N,3)
        Probabilités initiales pour [no, bottom, top].
    aligned : np.ndarray, shape (N,)
        Signal de peak (0 à 1) indiquant la probabilité d'un hit.
    threshold : float
        Seuil au‐delà duquel on considère qu'il y a un hit à booster.
    boost_factor : float
        Facteur de boost appliqué aux probabilités bottom/top.

    Returns
    -------
    y_boosted : np.ndarray, shape (N,3)
        Probabilités après boost et re‐normalisation.
    """
    y_boosted = y_pred_proba.copy()
    N = y_boosted.shape[0]
    assert aligned.shape[0] == N, "aligned doit avoir même longueur que y_pred_proba"

    # On booste bottom et top là où aligned dépasse le seuil
    mask = aligned > threshold
    y_boosted[mask, 1] *= boost_factor
    y_boosted[mask, 2] *= boost_factor

    # Re‐normalisation ligne par ligne
    sums = y_boosted.sum(axis=1, keepdims=True)
    # Pour éviter division par zéro
    sums[sums == 0] = 1.0
    y_boosted /= sums

    return y_boosted

    

In [20]:
max_dist=3
biais=0.5
ratio=0.4 
boost=1.5 
tresh=0.2


    
# 2) construire prob_peak
prob_peak = np.zeros_like(y_pred_proba_GRU_CNN)
# on répartit biais entre bottom et top selon beta
prob_peak[:, 1] = aligned_extend * biais   # bottom
prob_peak[:, 2] = aligned_extend * biais        # top
# la probabilité de no-hit est le reste
prob_peak[:, 0] = 1 - prob_peak[:, 1] - prob_peak[:, 2]
y_boosted = boosting_with_aligned(
            y_pred_proba_GRU_CNN, aligned_extend, tresh, boost
        )
# 3) fusion pondérée
fused = ratio * prob_peak + (1 - ratio) * y_boosted
# renormalisation ligne par ligne
fused /= fused.sum(axis=1, keepdims=True)
y_pred_fused = np.argmax(fused,axis=1)

In [21]:
print_class_metrics(y_test, y_pred_GRU_CNN)
c = confusion_matrix(y_test, y_pred_GRU_CNN, labels=[0,1,2])
print(c)
print_class_metrics(y_test, y_pred_fused)
c = confusion_matrix(y_test, y_pred_fused, labels=[0,1,2])
print(c)

Accuracy globale : 0.8683

Classe 0:
  support   = 191677
  precision = 0.966
  rappel    = 0.894
  f1-score  = 0.928

Classe 1:
  support   = 17566
  precision = 0.598
  rappel    = 0.749
  f1-score  = 0.665

Classe 2:
  support   = 17478
  precision = 0.453
  rappel    = 0.709
  f1-score  = 0.553

[[171318   7621  12738]
 [  2183  13159   2224]
 [  3859   1226  12393]]
Accuracy globale : 0.9074

Classe 0:
  support   = 191677
  precision = 0.948
  rappel    = 0.956
  f1-score  = 0.952

Classe 1:
  support   = 17566
  precision = 0.715
  rappel    = 0.674
  f1-score  = 0.694

Classe 2:
  support   = 17478
  precision = 0.633
  rappel    = 0.610
  f1-score  = 0.621

[[183226   3790   4661]
 [  4204  11833   1529]
 [  5882    930  10666]]


In [22]:
aligned_extend.sum()

np.float64(29726.684519150294)

In [23]:
df_comparaison = pd.DataFrame({
    'frappe': y_test,
    'son': aligned_extend,
    'top' : y_pred_proba_GRU_CNN[:,1],
    'bottom' : y_pred_proba_GRU_CNN[:,2],
    'fused_top' : fused[:,1],
    'fused_bottom' : fused[:,2],
    'pred' : y_pred_fused

})

from IPython.display import display, HTML
df_arrondi = df_comparaison.round(2)

# Affichage avec scroll
display(HTML(df_arrondi.to_html(max_rows=500, max_cols=7, notebook=True)))

,frappe,son,top,bottom,fused_top,fused_bottom,pred
0,0.0,0.00,0.04,0.03,0.02,0.02,0
1,0.0,0.00,0.04,0.03,0.02,0.02,0
2,0.0,0.00,0.04,0.03,0.02,0.02,0
3,0.0,0.00,0.04,0.03,0.02,0.02,0
4,0.0,0.00,0.04,0.03,0.02,0.02,0
5,0.0,0.00,0.04,0.03,0.02,0.02,0
6,0.0,0.00,0.04,0.03,0.02,0.02,0
7,0.0,0.00,0.04,0.03,0.02,0.02,0
8,0.0,0.00,0.04,0.03,0.02,0.02,0
9,0.0,0.00,0.04,0.03,0.02,0.02,0


In [24]:
y_fused_modif = traiter_sequence(y_pred_fused)

In [25]:
print_class_metrics(y_test, y_pred_fused)
c = confusion_matrix(y_test, y_pred_fused, labels=[0,1,2])
print(c)
print_class_metrics(y_test, y_fused_modif)
c = confusion_matrix(y_test, y_fused_modif, labels=[0,1,2])
print(c)

Accuracy globale : 0.9074

Classe 0:
  support   = 191677
  precision = 0.948
  rappel    = 0.956
  f1-score  = 0.952

Classe 1:
  support   = 17566
  precision = 0.715
  rappel    = 0.674
  f1-score  = 0.694

Classe 2:
  support   = 17478
  precision = 0.633
  rappel    = 0.610
  f1-score  = 0.621

[[183226   3790   4661]
 [  4204  11833   1529]
 [  5882    930  10666]]
Accuracy globale : 0.9114

Classe 0:
  support   = 191677
  precision = 0.940
  rappel    = 0.967
  f1-score  = 0.953

Classe 1:
  support   = 17566
  precision = 0.756
  rappel    = 0.633
  f1-score  = 0.689

Classe 2:
  support   = 17478
  precision = 0.683
  rappel    = 0.581
  f1-score  = 0.628

[[185365   2985   3327]
 [  5061  11115   1390]
 [  6722    606  10150]]


In [26]:
result = eval_detection(y_test, y_fused_modif, tol=3)
print(result)

{'series_total_y_true': 5841, 'series_total_y_pred': 4920, 'correct_label_match': 4123, 'wrong_label_match': 397, 'false_positives': 400, 'missed_true_blocks': 1322}


In [27]:
# # Définissez vos grilles de paramètres

# biais_vals    = [0.3, 0.5, 0.7]
# ratio_vals = [0.3,0.5,0.7]
# boost_factor_vals = [1,1.5,2,3,4]
# treshold_vals = [0.2,0.4,0.5,0.6]

# best_score = -1
# best_params = None
# for boost in boost_factor_vals:
#     for tresh in treshold_vals:
#         for max_dist in max_dist_vals:
#             for biais in biais_vals:
            
                    
#                 # 2) construire prob_peak
#                 prob_peak = np.zeros_like(y_pred_proba_GRU_CNN)
#                 # on répartit biais entre bottom et top selon beta
#                 prob_peak[:, 1] = aligned_extend * biais   # bottom
#                 prob_peak[:, 2] = aligned_extend * biais        # top
#                 # la probabilité de no-hit est le reste
#                 prob_peak[:, 0] = 1 - prob_peak[:, 1] - prob_peak[:, 2]
#                 y_boosted = boosting_with_aligned(
#                             y_pred_proba_GRU_CNN, aligned_extend, tresh, boost
#                         )
#                 # 3) fusion pondérée
#                 fused = ratio * prob_peak + (1 - ratio) * y_boosted
#                 # renormalisation ligne par ligne
#                 fused /= fused.sum(axis=1, keepdims=True)
#                 y_pred_fused = np.argmax(fused,axis=1)
#                 y_fused_modif = traiter_sequence(y_pred_fused)
#                 result = eval_detection(y_test, y_fused_modif, tol=3)
#                 print(result)
#                 score = result["correct_label_match"]
#                 if score > best_score:
#                     best_score = score
#                     best_params = (max_dist, biais,ratio,boost,tresh)
#                     print("Nouvel optimum !!!!!!!!!! pour max_dist : {max_dist}, biais :{biais}, ratio :{})

# print("\n=== Résultat final ===")
# print("Meilleur ", best_score)
# print("Paramètres optimaux :",
#       f"max_dist={best_params[0]}, biais={best_params[1]}, "
#       f"ratio={best_params[2]}",f"boost={best_params[3]}",f"tresh={best_params[4]}")

In [28]:
import numpy as np
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
from collections import OrderedDict

# --- 1) Définition de l’espace de recherche
space = {
    'biais':    hp.choice('biais',    [0.3,0.5,0.7,0.9,1.1,1.3,1.5,1.7,1.9]),
    'ratio':    hp.choice('ratio',    [0.2,0.25,0.3,0.35,0.4,0.45,0.5,0.55,0.6,0.65,0.7]),
    'boost':    hp.choice('boost',    [0.5,1,1.5,2,3,4]),
    'tresh':    hp.choice('tresh',    [0.35]),
    'max_dist': hp.choice('max_dist', [1,2,3,4]),
}

# --- 2) Fonction objectif qui retourne aussi les params
def objective(params):
    biais    = params['biais']
    ratio    = params['ratio']
    boost    = params['boost']
    tresh    = params['tresh']
    max_dist = params['max_dist']

    # Calcul de aligned_extend (exemple)
    aligned_extend = np.zeros_like(aligned)
    for i in range(len(aligned)):
        start = max(0, i - max_dist)
        end   = min(len(aligned), i + max_dist + 1)
        aligned_extend[i] = np.max(aligned[start:end])

    # Construction de prob_peak
    prob_peak = np.zeros_like(y_pred_proba_GRU_CNN)
    prob_peak[:,1] = aligned_extend * biais
    prob_peak[:,2] = aligned_extend * biais
    prob_peak[:,0] = 1 - prob_peak[:,1] - prob_peak[:,2]

    # Boosting et fusion
    y_boosted = boosting_with_aligned(y_pred_proba_GRU_CNN, aligned_extend, tresh, boost)
    fused = ratio * prob_peak + (1 - ratio) * y_boosted
    fused /= fused.sum(axis=1, keepdims=True)

    # Prédiction et évaluation
    y_pred_fused  = np.argmax(fused, axis=1)
    y_fused_modif = traiter_sequence(y_pred_fused)
    result = eval_detection(y_test, y_fused_modif, tol=max_dist)
    score  = result["correct_label_match"] - result["false_positives"]

    # On renvoie en plus les params pour pouvoir les lister ensuite
    return {
        'loss': -score,
        'status': STATUS_OK,
        'params': params,
    }

# --- 3) Lancement de l’optimisation
trials = Trials()
best = fmin(
    fn=objective,
    space=space,
    algo=tpe.suggest,
    max_evals=100,
    trials=trials
)



# 1) On construit un dict qui garde pour chaque param set unique le meilleur score observé
best_per_params = {}
for r in trials.results:
    params = r['params']
    # Pour pouvoir utiliser comme clé, on transforme en tuple trié
    key = tuple(sorted(params.items()))
    score = -r['loss']
    # On garde le meilleur score pour cet ensemble de params
    if key not in best_per_params or best_per_params[key] < score:
        best_per_params[key] = score

# 2) On trie ces ensembles uniques par score décroissant
sorted_unique = sorted(best_per_params.items(), key=lambda x: -x[1])

# 3) On affiche les 10 premiers
print("\nTop 10 des ensembles de paramètres DISTINCTS :")
for rank, (param_tuple, score) in enumerate(sorted_unique[:10], start=1):
    params_dict = dict(param_tuple)
    print(f"{rank:2d}. score = {score:.4f}  →  params = {params_dict}")


  import pkg_resources



100%|██████████| 100/100 [05:14<00:00,  3.14s/trial, best loss: -3869.0]

Top 10 des ensembles de paramètres DISTINCTS :
 1. score = 3869.0000  →  params = {'biais': 0.7, 'boost': 0.5, 'max_dist': 4, 'ratio': 0.3, 'tresh': 0.35}
 2. score = 3842.0000  →  params = {'biais': 0.3, 'boost': 1.5, 'max_dist': 4, 'ratio': 0.3, 'tresh': 0.35}
 3. score = 3833.0000  →  params = {'biais': 0.5, 'boost': 0.5, 'max_dist': 4, 'ratio': 0.3, 'tresh': 0.35}
 4. score = 3820.0000  →  params = {'biais': 0.7, 'boost': 0.5, 'max_dist': 4, 'ratio': 0.25, 'tresh': 0.35}
 5. score = 3727.0000  →  params = {'biais': 0.7, 'boost': 0.5, 'max_dist': 4, 'ratio': 0.2, 'tresh': 0.35}
 6. score = 3555.0000  →  params = {'biais': 0.9, 'boost': 0.5, 'max_dist': 4, 'ratio': 0.3, 'tresh': 0.35}
 7. score = 3507.0000  →  params = {'biais': 0.7, 'boost': 1, 'max_dist': 4, 'ratio': 0.3, 'tresh': 0.35}
 8. score = 3385.0000  →  params = {'biais': 0.7, 'boost': 1, 'max_dist': 4, 'ratio': 0.35, 'tresh': 0.35}
 9. score = 3380.

In [29]:
y_pred_GRU_CNN[-25191:].shape

(25191,)

Test sur des vidéo seule

In [30]:
import numpy as np
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
from collections import OrderedDict


# --- 1) Définition de l’espace de recherche
space = {
    'biais':    hp.choice('biais',    [0.3,0.5,0.7,0.9,1.1,1.3,1.5,1.7,1.9]),
    'ratio':    hp.choice('ratio',    [0.2,0.25,0.3,0.35,0.4,0.45,0.5,0.55,0.6,0.65,0.7]),
    'boost':    hp.choice('boost',    [0.5,1,1.5,2,3,4]),
    'tresh':    hp.choice('tresh',    [0.35]),
    'max_dist': hp.choice('max_dist', [1,2,3,4]),
}

# --- 2) Fonction objectif qui retourne aussi les params
def objective(params):
    biais    = params['biais']
    ratio    = params['ratio']
    boost    = params['boost']
    tresh    = params['tresh']
    max_dist = params['max_dist']

    # Calcul de aligned_extend (exemple)
    aligned_extend = np.zeros_like(aligned)
    for i in range(len(aligned)):
        start = max(0, i - max_dist)
        end   = min(len(aligned), i + max_dist + 1)
        aligned_extend[i] = np.max(aligned[start:end])

    # Construction de prob_peak
    prob_peak = np.zeros_like(y_pred_proba_GRU_CNN[5736 : 7884])
    prob_peak[:,1] = aligned_extend[5736 : 7884]* biais
    prob_peak[:,2] = aligned_extend[5736 : 7884]* biais
    prob_peak[:,0] = 1 - prob_peak[:,1] - prob_peak[:,2]

    # Boosting et fusion
    y_boosted = boosting_with_aligned(y_pred_proba_GRU_CNN[5736 : 7884], aligned_extend[5736 : 7884], tresh, boost)
    fused = ratio * prob_peak + (1 - ratio) * y_boosted
    fused /= fused.sum(axis=1, keepdims=True)

    # Prédiction et évaluation
    y_pred_fused  = np.argmax(fused, axis=1)
    y_fused_modif = traiter_sequence(y_pred_fused)
    result = eval_detection(y_test[5736 : 7884], y_fused_modif, tol=max_dist)
    score  = result["correct_label_match"] - result["false_positives"]

    # On renvoie en plus les params pour pouvoir les lister ensuite
    return {
        'loss': -score,
        'status': STATUS_OK,
        'params': params,
    }

# --- 3) Lancement de l’optimisation
trials = Trials()
best = fmin(
    fn=objective,
    space=space,
    algo=tpe.suggest,
    max_evals=100,
    trials=trials
)



# 1) On construit un dict qui garde pour chaque param set unique le meilleur score observé
best_per_params = {}
for r in trials.results:
    params = r['params']
    # Pour pouvoir utiliser comme clé, on transforme en tuple trié
    key = tuple(sorted(params.items()))
    score = -r['loss']
    # On garde le meilleur score pour cet ensemble de params
    if key not in best_per_params or best_per_params[key] < score:
        best_per_params[key] = score

# 2) On trie ces ensembles uniques par score décroissant
sorted_unique = sorted(best_per_params.items(), key=lambda x: -x[1])

# 3) On affiche les 10 premiers
print("\nTop 10 des ensembles de paramètres DISTINCTS :")
for rank, (param_tuple, score) in enumerate(sorted_unique[:10], start=1):
    params_dict = dict(param_tuple)
    print(f"{rank:2d}. score = {score:.4f}  →  params = {params_dict}")

 46%|████▌     | 46/100 [00:41<00:48,  1.11trial/s, best loss: -40.0]


KeyboardInterrupt: 

In [ ]:
2324 +3412 + 2148


7884

In [ ]:
biais= 0.35
boost= 1.5
max_dist= 4
ratio= 0.3
tresh= 0.35

# biais= 0.25
# boost= 3
# max_dist= 3
# ratio= 0.4
# tresh= 0.35

# Calcul de aligned_extend (exemple)
aligned_extend = np.zeros_like(aligned)
for i in range(len(aligned)):
    start = max(0, i - max_dist)
    end   = min(len(aligned), i + max_dist + 1)
    aligned_extend[i] = np.max(aligned[start:end])

# Construction de prob_peak
prob_peak = np.zeros_like(y_pred_proba_GRU_CNN[5736 : 7884])
prob_peak[:,1] = aligned_extend[5736 : 7884] * biais
prob_peak[:,2] = aligned_extend[5736 : 7884] * biais
prob_peak[:,0] = 1 - prob_peak[:,1] - prob_peak[:,2]

# Boosting et fusion
y_boosted = boosting_with_aligned(y_pred_proba_GRU_CNN[5736 : 7884], aligned_extend[5736 : 7884], tresh, boost)
fused = ratio * prob_peak + (1 - ratio) * y_boosted
fused /= fused.sum(axis=1, keepdims=True)

# Prédiction et évaluation
y_pred_fused  = np.argmax(fused, axis=1)
y_fused_modif = traiter_sequence(y_pred_fused)

In [ ]:
print_class_metrics(y_test[5736 : 7884], y_fused_modif)
c = confusion_matrix(y_test[5736 : 7884], y_fused_modif, labels=[0,1,2])
print(c)


print_class_metrics(y_test[5736 : 7884], y_pred_fused)
c = confusion_matrix(y_test[5736 : 7884], y_pred_fused, labels=[0,1,2])
print(c)

print_class_metrics(y_test[5736 : 7884], y_pred_GRU_CNN[5736 : 7884])
c = confusion_matrix(y_test[5736 : 7884], y_pred_GRU_CNN[5736 : 7884], labels=[0,1,2])
print(c)

Accuracy globale : 0.9092

Classe 0:
  support   = 1782
  precision = 0.940
  rappel    = 0.971
  f1-score  = 0.955

Classe 1:
  support   = 186
  precision = 0.731
  rappel    = 0.613
  f1-score  = 0.667

Classe 2:
  support   = 180
  precision = 0.720
  rappel    = 0.600
  f1-score  = 0.655

[[1731   27   24]
 [  54  114   18]
 [  57   15  108]]
Accuracy globale : 0.9055

Classe 0:
  support   = 1782
  precision = 0.963
  rappel    = 0.940
  f1-score  = 0.951

Classe 1:
  support   = 186
  precision = 0.668
  rappel    = 0.726
  f1-score  = 0.696

Classe 2:
  support   = 180
  precision = 0.655
  rappel    = 0.750
  f1-score  = 0.699

[[1675   55   52]
 [  32  135   19]
 [  33   12  135]]
Accuracy globale : 0.8454

Classe 0:
  support   = 1782
  precision = 0.962
  rappel    = 0.865
  f1-score  = 0.911

Classe 1:
  support   = 186
  precision = 0.532
  rappel    = 0.769
  f1-score  = 0.629

Classe 2:
  support   = 180
  precision = 0.477
  rappel    = 0.733
  f1-score  = 0.578

[[154

Test Best Trial

In [31]:
biais= 0.7 
boost= 0.5
max_dist= 4
ratio= 0.3
tresh= 0.35
# Calcul de aligned_extend (exemple)
aligned_extend = np.zeros_like(aligned)
for i in range(len(aligned)):
    start = max(0, i - max_dist)
    end   = min(len(aligned), i + max_dist + 1)
    aligned_extend[i] = np.max(aligned[start:end])

# Construction de prob_peak
prob_peak = np.zeros_like(y_pred_proba_GRU_CNN)
prob_peak[:,1] = aligned_extend * biais
prob_peak[:,2] = aligned_extend * biais
prob_peak[:,0] = 1 - prob_peak[:,1] - prob_peak[:,2]

# Boosting et fusion
y_boosted = boosting_with_aligned(y_pred_proba_GRU_CNN, aligned_extend, tresh, boost)
fused = ratio * prob_peak + (1 - ratio) * y_boosted
fused /= fused.sum(axis=1, keepdims=True)

# Prédiction et évaluation
y_pred_fused  = np.argmax(fused, axis=1)
y_fused_modif = traiter_sequence(y_pred_fused)

In [32]:
# print_class_metrics(y_test, y_fused_modif)
# c = confusion_matrix(y_test, y_fused_modif, labels=[0,1,2])
# print(c)


print_class_metrics(y_test, y_pred_fused)
c = confusion_matrix(y_test, y_pred_fused, labels=[0,1,2])
print(c)

print_class_metrics(y_test, y_pred_GRU_CNN)
c = confusion_matrix(y_test, y_pred_GRU_CNN, labels=[0,1,2])
print(c)

Accuracy globale : 0.8973

Classe 0:
  support   = 191677
  precision = 0.961
  rappel    = 0.933
  f1-score  = 0.947

Classe 1:
  support   = 17566
  precision = 0.664
  rappel    = 0.730
  f1-score  = 0.695

Classe 2:
  support   = 17478
  precision = 0.553
  rappel    = 0.679
  f1-score  = 0.610

[[178744   5339   7594]
 [  2749  12826   1991]
 [  4450   1161  11867]]
Accuracy globale : 0.8683

Classe 0:
  support   = 191677
  precision = 0.966
  rappel    = 0.894
  f1-score  = 0.928

Classe 1:
  support   = 17566
  precision = 0.598
  rappel    = 0.749
  f1-score  = 0.665

Classe 2:
  support   = 17478
  precision = 0.453
  rappel    = 0.709
  f1-score  = 0.553

[[171318   7621  12738]
 [  2183  13159   2224]
 [  3859   1226  12393]]


In [ ]:
import pandas as pd
from IPython.display import display, HTML

# Charger le fichier
df_strike_preds = pd.read_csv("/home/onyxia/work/DATA/TrackNet/out/strike_preds.csv")

# Arrondir (si souhaité)
df_arrondi = df_strike_preds.round(2)

# Affichage avec scroll
display(HTML(df_arrondi.to_html(max_rows=500, max_cols=7, notebook=True)))

,Frame,Strike_Pred,peak_prob,traj_prob_top,traj_prob_bottom
0,0,0,0.00,0.13,0.56
1,1,0,0.00,0.02,0.23
2,2,0,0.43,0.04,0.64
3,3,0,0.43,0.06,0.15
4,4,0,0.43,0.05,0.08
5,5,0,0.43,0.15,0.06
6,6,0,0.55,0.13,0.02
7,7,0,0.55,0.16,0.01
8,8,0,0.55,0.49,0.00
9,9,0,0.55,0.14,0.00
